In [6]:
# ================================================================
#  STANDALONE SINGLE-FRAME LATENCY BENCHMARK — 5 SNIPPETS
#
#  Snippet selection (per Chapter 4 methodology):
#    Snippet 1: Engaged class (DAiSEE engagement level 3)
#    Snippet 2: Passive class (DAiSEE engagement level 2)
#    Snippet 3: Disengaged class (DAiSEE engagement level 0 or 1)
#    Snippet 4: Bright illumination condition
#    Snippet 5: Dark illumination condition
#
#  Each snippet: 20 warm-up + 100 timed single-frame inferences
#  Total: 500 timed trials across 5 edge-case snippets
#  No training, no evaluation — latency numbers only
# ================================================================
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import time, warnings, sys, glob, platform, subprocess
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import cv2
import mediapipe as mp
from deepface import DeepFace
from scipy import stats as scipy_stats

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────
MODEL_PATH     = "/Users/mike/Desktop/Praxis/DAiSEE/SMOTE/model_T10.pt"
TEST_VIDEO_DIR = "/Users/mike/Desktop/Praxis/DAiSEE/DataSet/Test"
TEST_LABEL_CSV = "/Users/mike/Desktop/Praxis/DAiSEE/Labels/TestLabels.csv"

# ── Hyper-parameters (must match training) ─────────────────────
T           = 10
DIM_INPUT   = 522
D_MODEL     = 512
NUM_HEADS   = 8
NUM_LAYERS  = 2
NUM_CLASSES = 3
DROPOUT     = 0.3

# Smoke test: 3 / 5 / 50.   Real run: 20 / 100 / 200.
N_WARMUP            = 20
N_TIMED             = 100
N_BRIGHTNESS_SAMPLE = 200

TARGET = 130.0   # latency budget, ms

# ── Hardware auto-detection (never hardcode this) ──────────────
def detect_cpu():
    try:
        return subprocess.check_output(
            ["sysctl", "-n", "machdep.cpu.brand_string"], text=True
        ).strip()
    except Exception:
        return platform.processor() or "unknown CPU"

CPU_NAME = detect_cpu()
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
BACKEND = "MPS (Metal GPU)" if DEVICE.type == "mps" else "CPU"
HARDWARE_STR = f"{CPU_NAME}, {BACKEND}"

print(f"Device: {DEVICE}")
print(f"Hardware: {HARDWARE_STR}")
print(f"Machine: {platform.machine()}  |  Python: {platform.python_version()}")
print(f"torch: {torch.__version__}  |  MPS available: {torch.backends.mps.is_available()}")

# ================================================================
#  STEP 1: SELECT 5 SNIPPETS COVERING DISTINCT EDGE CASES
# ================================================================
try:
    labels_df = pd.read_csv(TEST_LABEL_CSV)
    print(f"\nLoaded labels: {len(labels_df)} rows")
    print(f"Columns: {list(labels_df.columns)}")
    print(labels_df.head())
except FileNotFoundError:
    print(f"ERROR: Label file not found at {TEST_LABEL_CSV}")
    print("Searching for label files...")
    for f in glob.glob("/Users/mike/Desktop/Praxis/DAiSEE/**/*est*abel*", recursive=True):
        print(f"  Found: {f}")
    sys.exit(1)

# ── Identify the correct column names ──────────────────────────
eng_col = None
clip_col = None
for c in labels_df.columns:
    if 'engag' in c.lower():
        eng_col = c
    if 'clip' in c.lower() or c.lower() == 'clipid':
        clip_col = c

if eng_col is None or clip_col is None:
    print(f"\nCould not auto-detect columns: {labels_df.columns.tolist()}")
    sys.exit(1)

print(f"\nUsing columns: clip='{clip_col}', engagement='{eng_col}'")

# ── Map engagement levels to tri-state ─────────────────────────
labels_df["tri_state"] = labels_df[eng_col].map(
    {0: "Disengaged", 1: "Disengaged", 2: "Passive", 3: "Engaged"}
)

# ── Build a filename index once (fast lookup) ──────────────────
print("\nIndexing test videos...")
VIDEO_INDEX = {}
for root, _, files in os.walk(TEST_VIDEO_DIR):
    for f in files:
        if f.lower().endswith((".avi", ".mp4")):
            full = os.path.join(root, f)
            VIDEO_INDEX[f] = full
            VIDEO_INDEX[os.path.splitext(f)[0]] = full
print(f"  Indexed {len(set(VIDEO_INDEX.values()))} videos")

def find_video_path(clip_id):
    key = str(clip_id).strip()
    if key in VIDEO_INDEX:
        return VIDEO_INDEX[key]
    return VIDEO_INDEX.get(os.path.splitext(key)[0])

# ── Helper: compute frame brightness ──────────────────────────
def get_frame_brightness(video_path):
    cap = cv2.VideoCapture(video_path)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        return None
    return float(np.mean(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)))

# ── Select one snippet per engagement class ────────────────────
print("\n" + "="*60)
print("  SELECTING 5 EDGE-CASE SNIPPETS")
print("="*60)

selected_snippets = []

engaged_rows    = labels_df[labels_df["tri_state"] == "Engaged"]
passive_rows    = labels_df[labels_df["tri_state"] == "Passive"]
disengaged_rows = labels_df[labels_df["tri_state"] == "Disengaged"]
print(f"\nEngaged candidates:    {len(engaged_rows)}")
print(f"Passive candidates:    {len(passive_rows)}")
print(f"Disengaged candidates: {len(disengaged_rows)}")

for label, rows, snippet_name in [
    ("Engaged",    engaged_rows,    "Snippet 1: Engaged"),
    ("Passive",    passive_rows,    "Snippet 2: Passive"),
    ("Disengaged", disengaged_rows, "Snippet 3: Disengaged"),
]:
    found = False
    indices = list(rows.index)
    mid = len(indices) // 2
    # centre-outward search order
    search_order = sorted(indices, key=lambda ix: abs(indices.index(ix) - mid))

    for idx in search_order:
        clip_id = rows.loc[idx, clip_col]
        vpath = find_video_path(clip_id)
        if vpath is None:
            continue
        cap = cv2.VideoCapture(vpath)
        ret, frame = cap.read()
        cap.release()
        if ret and frame is not None:
            brightness = get_frame_brightness(vpath)
            selected_snippets.append({
                "name": snippet_name,
                "clip_id": clip_id,
                "path": vpath,
                "class": label,
                "engagement_level": rows.loc[idx, eng_col],
                "brightness": brightness,
            })
            print(f"  {snippet_name}: {clip_id} (level={rows.loc[idx, eng_col]}, "
                  f"brightness={brightness:.1f})")
            found = True
            break
    if not found:
        print(f"  WARNING: Could not find valid video for {label}")

# ── Snippet 4 & 5: Bright and Dark illumination extremes ──────
print(f"\nScanning {N_BRIGHTNESS_SAMPLE} test videos for illumination extremes...")

all_test_clips = labels_df[clip_col].tolist()
np.random.seed(42)
sample_clips = np.random.choice(
    all_test_clips, size=min(N_BRIGHTNESS_SAMPLE, len(all_test_clips)), replace=False
)

brightness_data = []
for clip_id in sample_clips:
    vpath = find_video_path(clip_id)
    if vpath:
        b = get_frame_brightness(vpath)
        if b is not None:
            brightness_data.append({"clip_id": clip_id, "path": vpath, "brightness": b})

if len(brightness_data) < 2:
    print("  ERROR: Not enough readable videos for brightness extremes.")
    sys.exit(1)

brightness_data.sort(key=lambda x: x["brightness"])
print(f"  Scanned {len(brightness_data)} videos")
print(f"  Brightness range: {brightness_data[0]['brightness']:.1f} (darkest) "
      f"to {brightness_data[-1]['brightness']:.1f} (brightest)")

already_selected_paths = {s["path"] for s in selected_snippets}

bright = brightness_data[-1]
for item in reversed(brightness_data):
    if item["path"] not in already_selected_paths:
        bright = item
        break

selected_snippets.append({
    "name": "Snippet 4: Bright illumination",
    "clip_id": bright["clip_id"], "path": bright["path"],
    "class": "Bright", "engagement_level": "-", "brightness": bright["brightness"],
})
print(f"  Snippet 4 (Bright): {bright['clip_id']} (brightness={bright['brightness']:.1f})")

dark = brightness_data[0]
for item in brightness_data:
    if item["path"] not in already_selected_paths and item["path"] != bright["path"]:
        dark = item
        break

selected_snippets.append({
    "name": "Snippet 5: Dark illumination",
    "clip_id": dark["clip_id"], "path": dark["path"],
    "class": "Dark", "engagement_level": "-", "brightness": dark["brightness"],
})
print(f"  Snippet 5 (Dark):   {dark['clip_id']} (brightness={dark['brightness']:.1f})")

# ── Summary of selected snippets ──────────────────────────────
print(f"\n{'='*60}")
print(f"  SELECTED SNIPPETS SUMMARY")
print(f"{'='*60}")
print(f"  {'#':<4} {'Name':<32} {'ClipID':<18} {'Level':<6} {'Bright':>7}")
print(f"  {'-'*4} {'-'*32} {'-'*18} {'-'*6} {'-'*7}")
for i, s in enumerate(selected_snippets):
    print(f"  {i+1:<4} {s['name']:<32} {str(s['clip_id']):<18} "
          f"{str(s['engagement_level']):<6} {s['brightness']:>7.1f}")

# ================================================================
#  STEP 2: MODEL DEFINITION AND LOADING
# ================================================================
class HybridEngagementModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fusion_proj = nn.Linear(DIM_INPUT, D_MODEL)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=NUM_HEADS,
            dim_feedforward=D_MODEL * 4, dropout=DROPOUT,
            activation="gelu", batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=NUM_LAYERS)
        self.classifier = nn.Linear(D_MODEL, NUM_CLASSES)

    def forward(self, x):
        x = self.fusion_proj(x)
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.classifier(x)

model = HybridEngagementModel().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print(f"\nModel loaded from {MODEL_PATH}")

mp_face = mp.solutions.face_mesh.FaceMesh(
    static_image_mode=True, max_num_faces=1,
    refine_landmarks=True, min_detection_confidence=0.5,
)

# ── Helper: geometric features ─────────────────────────────────
def extract_geometric(landmarks, img_w, img_h):
    def pt(idx):
        lm = landmarks[idx]
        return np.array([lm.x * img_w, lm.y * img_h, lm.z * img_w])
    def ear(eye_ids):
        p1, p2, p3, p4, p5, p6 = [pt(i) for i in eye_ids]
        num = np.linalg.norm(p2 - p6) + np.linalg.norm(p3 - p5)
        den = 2.0 * np.linalg.norm(p1 - p4)
        return num / (den + 1e-6)
    ear_l = ear([33, 160, 158, 133, 153, 144])
    ear_r = ear([362, 385, 387, 263, 373, 380])
    ear_avg = (ear_l + ear_r) / 2.0
    p_m = [pt(i) for i in [61, 291, 39, 181, 0, 17]]
    mar = (np.linalg.norm(p_m[2] - p_m[3]) + np.linalg.norm(p_m[4] - p_m[5])) / \
          (2.0 * np.linalg.norm(p_m[0] - p_m[1]) + 1e-6)
    nose  = pt(1);  chin  = pt(152)
    l_eye = pt(33); r_eye = pt(263)
    face_vec = chin - nose
    pitch = np.arctan2(face_vec[1], face_vec[2])
    yaw   = np.arctan2(face_vec[0], face_vec[2])
    roll  = np.arctan2(r_eye[1] - l_eye[1], r_eye[0] - l_eye[0])
    ied   = np.linalg.norm(l_eye[:2] - r_eye[:2]) / (img_w + 1e-6)
    return np.array([ear_l, ear_r, ear_avg, mar, pitch, yaw, roll, ied,
                     nose[0] / img_w, nose[1] / img_h], dtype=np.float32)

# ── Single-frame pipeline ──────────────────────────────────────
def run_single_frame(frame_rgb):
    h, w = frame_rgb.shape[:2]
    # 1) MediaPipe
    t0 = time.perf_counter()
    result = mp_face.process(frame_rgb)
    t_mp = (time.perf_counter() - t0) * 1000
    if result.multi_face_landmarks:
        geo = extract_geometric(result.multi_face_landmarks[0].landmark, w, h)
    else:
        geo = np.zeros(10, dtype=np.float32)
    # 2) GhostFaceNet
    t0 = time.perf_counter()
    emb = DeepFace.represent(frame_rgb, model_name="GhostFaceNet",
                             enforce_detection=False, detector_backend="skip"
                             )[0]["embedding"]
    emb = np.array(emb, dtype=np.float32)
    t_gf = (time.perf_counter() - t0) * 1000
    # 3) Transformer
    feat = np.concatenate([emb, geo])
    seq  = np.tile(feat, (T, 1))
    x    = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = model(x)
    if DEVICE.type == "mps":
        torch.mps.synchronize()
    t_tf = (time.perf_counter() - t0) * 1000
    return t_mp, t_gf, t_tf

# ================================================================
#  STEP 3: BENCHMARK ALL 5 SNIPPETS
# ================================================================
all_snippet_results = []
bench_t0 = time.perf_counter()

for snippet in selected_snippets:
    print(f"\n{'='*60}")
    print(f"  {snippet['name']} — ClipID: {snippet['clip_id']}")
    print(f"  Class: {snippet['class']}  |  Brightness: {snippet['brightness']:.1f}")
    print(f"  Warm-up: {N_WARMUP}  |  Timed: {N_TIMED}")
    print(f"{'='*60}")

    cap = cv2.VideoCapture(snippet["path"])
    ret, frame = cap.read()
    cap.release()
    if not ret:
        print(f"  ERROR: Cannot read frame — SKIPPING")
        continue
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    print(f"  Frame shape: {frame.shape}")

    print(f"  Warming up...", end=" ", flush=True)
    for _ in range(N_WARMUP):
        _ = run_single_frame(frame_rgb)
    print("done.")

    times_mp, times_gf, times_tf = [], [], []
    print(f"  Timing:", end=" ", flush=True)
    for i in range(N_TIMED):
        a, b, c = run_single_frame(frame_rgb)
        times_mp.append(a); times_gf.append(b); times_tf.append(c)
        if (i + 1) % 25 == 0:
            print(f"{i+1}", end=" ", flush=True)
    print("done.")

    times_mp = np.array(times_mp)
    times_gf = np.array(times_gf)
    times_tf = np.array(times_tf)
    times_total = times_mp + times_gf + times_tf

    res = {
        "snippet": snippet, "mp": times_mp, "gf": times_gf, "tf": times_tf,
        "total": times_total,
        "mean": np.mean(times_total), "median": np.median(times_total),
        "std": np.std(times_total), "min": np.min(times_total),
        "max": np.max(times_total),
        "p95": np.percentile(times_total, 95),
        "p99": np.percentile(times_total, 99),
    }
    all_snippet_results.append(res)

    print(f"  Mean={res['mean']:.2f}  Median={res['median']:.2f}  "
          f"Std={res['std']:.2f}  P95={res['p95']:.2f}  P99={res['p99']:.2f} ms")

if not all_snippet_results:
    print("\nERROR: No snippets benchmarked.")
    sys.exit(1)

print(f"\n  Total benchmark wall time: {(time.perf_counter()-bench_t0)/60:.1f} min")

# ================================================================
#  STEP 4: PER-SNIPPET SUMMARY TABLE
# ================================================================
print(f"\n\n{'='*80}")
print(f"  PER-SNIPPET LATENCY SUMMARY  ({len(all_snippet_results)} snippets x {N_TIMED} runs)")
print(f"{'='*80}")
print(f"  {'Snippet':<32} {'Mean':>7} {'Median':>7} {'Std':>7} "
      f"{'Min':>7} {'Max':>7} {'P95':>7} {'P99':>7}  {'Budget':>6}")
print(f"  {'-'*32} {'-'*7} {'-'*7} {'-'*7} {'-'*7} {'-'*7} {'-'*7} {'-'*7}  {'-'*6}")
for r in all_snippet_results:
    p95_pass = "PASS" if r["p95"] < TARGET else "FAIL"
    print(f"  {r['snippet']['name']:<32} {r['mean']:>7.2f} {r['median']:>7.2f} "
          f"{r['std']:>7.2f} {r['min']:>7.2f} {r['max']:>7.2f} "
          f"{r['p95']:>7.2f} {r['p99']:>7.2f}  {p95_pass:>6}")

# ================================================================
#  STEP 5: CONTENT-INVARIANCE TEST (Kruskal-Wallis)
# ================================================================
latency_groups = [r["total"] for r in all_snippet_results]
medians = [np.median(g) for g in latency_groups]
spread = max(medians) - min(medians)
kw_stat, kw_p = np.nan, np.nan

if len(latency_groups) >= 3:
    kw_stat, kw_p = scipy_stats.kruskal(*latency_groups)
    print(f"\n  Content-Invariance Test (Kruskal-Wallis H-test):")
    print(f"    H-statistic: {kw_stat:.4f}")
    print(f"    p-value:     {kw_p:.4f}")
    print(f"    Median range: {min(medians):.2f} – {max(medians):.2f} ms "
          f"(spread = {spread:.2f} ms)")
    if kw_p > 0.05:
        print(f"    Conclusion:  p > 0.05 — No significant difference across snippets.")
        print(f"                 Inference latency is CONTENT-INVARIANT.")
    else:
        print(f"    Conclusion:  p <= 0.05 — Statistically significant difference detected.")
        if spread < 10:
            print(f"    Practical significance: spread < 10ms — negligible for real-time operation.")
        else:
            print(f"    Practical significance: spread >= 10ms — warrants further investigation.")

# ================================================================
#  STEP 6: AGGREGATE STATISTICS
# ================================================================
all_total = np.concatenate([r["total"] for r in all_snippet_results])
all_mp    = np.concatenate([r["mp"]    for r in all_snippet_results])
all_gf    = np.concatenate([r["gf"]    for r in all_snippet_results])
all_tf    = np.concatenate([r["tf"]    for r in all_snippet_results])

def print_stats(arr, name):
    print(f"  {name:20s}  mean={np.mean(arr):7.2f}  median={np.median(arr):7.2f}  "
          f"std={np.std(arr):6.2f}  min={np.min(arr):7.2f}  max={np.max(arr):7.2f}  "
          f"P95={np.percentile(arr,95):7.2f}  P99={np.percentile(arr,99):7.2f} ms")

print(f"\n{'='*80}")
print(f"  AGGREGATE RESULTS  ({len(all_snippet_results)} snippets x {N_TIMED} runs "
      f"= {len(all_total)} total trials)")
print(f"{'='*80}")
print_stats(all_mp,    "MediaPipe")
print_stats(all_gf,    "GhostFaceNet")
print_stats(all_tf,    "Transformer")
print_stats(all_total, "TOTAL")

agg_mean = np.mean(all_total)
print(f"\n  Component breakdown (% of total):")
print(f"    MediaPipe:    {np.mean(all_mp)/agg_mean*100:5.1f}%")
print(f"    GhostFaceNet: {np.mean(all_gf)/agg_mean*100:5.1f}%")
print(f"    Transformer:  {np.mean(all_tf)/agg_mean*100:5.1f}%")

snippet_means = np.array([r["mean"] for r in all_snippet_results])
print(f"\n  Cross-snippet stability:")
for r in all_snippet_results:
    print(f"    {r['snippet']['name']:<32}: {r['mean']:.2f} ms")
print(f"    Mean of means:  {np.mean(snippet_means):.2f} ms")
print(f"    Std of means:   {np.std(snippet_means):.2f} ms")
print(f"    CV (coefficient of variation): "
      f"{np.std(snippet_means)/np.mean(snippet_means)*100:.1f}%")

# ── Pass / Fail against latency budget ───────────────────────
print(f"\n  Latency budget: {TARGET:.0f} ms (aggregate, {len(all_total)} trials)")
for label, val in [("Mean",   np.mean(all_total)),
                   ("Median", np.median(all_total)),
                   ("P95",    np.percentile(all_total, 95)),
                   ("P99",    np.percentile(all_total, 99))]:
    status = "PASS" if val < TARGET else "FAIL"
    print(f"    {label:7s}: {val:7.2f} ms — {status}")

print(f"\n  Runs under threshold (out of {len(all_total)}):")
for thr in [100, 130, 150, 200]:
    count = int(np.sum(all_total < thr))
    print(f"    < {thr:4d} ms: {count:4d}/{len(all_total)}  "
          f"({count/len(all_total)*100:5.1f}%)")

# ================================================================
#  STEP 7: FINAL SUMMARY
# ================================================================
print(f"\n{'='*80}")
print(f"  FINAL SUMMARY")
print(f"{'='*80}")
print(f"  Snippets tested:        {len(all_snippet_results)}")
print(f"  Runs per snippet:       {N_TIMED}")
print(f"  Total trials:           {len(all_total)}")
print(f"  Hardware:               {HARDWARE_STR}")
print(f"  torch / Python:         {torch.__version__} / {platform.python_version()}")
print(f"")
print(f"  Aggregate latency:      {np.mean(all_total):.2f} ± "
      f"{np.std(snippet_means):.2f} ms (mean ± cross-snippet std)")
print(f"  Aggregate P95:          {np.percentile(all_total, 95):.2f} ms")
print(f"  Aggregate P99:          {np.percentile(all_total, 99):.2f} ms")
print(f"  Latency budget (P95):   "
      f"{'PASS' if np.percentile(all_total, 95) < TARGET else 'FAIL'}")
print(f"  Content-invariant:      "
      f"{'YES' if (kw_p > 0.05 or spread < 10) else 'NO'}")
print(f"")
print(f"  Chapter 4 report line:")
print(f"  \"Across {len(all_total)} trials ({len(all_snippet_results)} edge-case "
      f"snippets x {N_TIMED} runs), the pipeline achieved a mean latency of "
      f"{np.mean(all_total):.2f} ± {np.std(snippet_means):.2f} ms with P95 = "
      f"{np.percentile(all_total, 95):.2f} ms, against a {TARGET:.0f} ms latency "
      f"budget, on {HARDWARE_STR}.\"")
print(f"{'='*80}")


Device: mps
Hardware: Apple M4, MPS (Metal GPU)
Machine: arm64  |  Python: 3.11.14
torch: 2.10.0  |  MPS available: True

Loaded labels: 1784 rows
Columns: ['ClipID', 'Boredom', 'Engagement', 'Confusion', 'Frustration ']
           ClipID  Boredom  Engagement  Confusion  Frustration 
0  5000441001.avi        1           2          0             0
1  5000441002.avi        0           2          0             0
2  5000441003.avi        1           2          0             0
3  5000441005.avi        2           2          0             0
4  5000441006.avi        2           2          1             2

Using columns: clip='ClipID', engagement='Engagement'

Indexing test videos...
  Indexed 1866 videos

  SELECTING 5 EDGE-CASE SNIPPETS

Engaged candidates:    814
Passive candidates:    882
Disengaged candidates: 88
  Snippet 1: Engaged: 5100451035.avi (level=3, brightness=131.2)
  Snippet 2: Passive: 5100401039.avi (level=2, brightness=139.5)
  Snippet 3: Disengaged: 5100462003.avi (level=1

I0000 00:00:1785966156.840006       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4


done.
  Timing: 25 50 75 100 done.
  Mean=108.16  Median=105.65  Std=22.84  P95=110.86  P99=121.97 ms

  Snippet 2: Passive — ClipID: 5100401039.avi
  Class: Passive  |  Brightness: 139.5
  Warm-up: 20  |  Timed: 100
  Frame shape: (480, 640, 3)
  Warming up... done.
  Timing: 25 50 75 100 done.
  Mean=113.94  Median=107.89  Std=31.30  P95=133.02  P99=300.41 ms

  Snippet 3: Disengaged — ClipID: 5100462003.avi
  Class: Disengaged  |  Brightness: 139.4
  Warm-up: 20  |  Timed: 100
  Frame shape: (480, 640, 3)
  Warming up... done.
  Timing: 25 50 75 100 done.
  Mean=111.84  Median=109.00  Std=20.37  P95=118.52  P99=132.28 ms

  Snippet 4: Bright illumination — ClipID: 8264120146.avi
  Class: Bright  |  Brightness: 160.9
  Warm-up: 20  |  Timed: 100
  Frame shape: (480, 640, 3)
  Warming up... done.
  Timing: 25 50 75 100 done.
  Mean=108.56  Median=108.50  Std=2.84  P95=113.88  P99=115.84 ms

  Snippet 5: Dark illumination — ClipID: 9070011010.mp4
  Class: Dark  |  Brightness: 88.2
  Wa